# D17 — Loan Disbursement Trend Analysis

ABES Engineering College | MCA 2nd Year | Data Analyst skill track

This reproducible notebook uses synthetic transaction data by default. A successful disbursement is defined as a successful credit transaction; validate this definition with real source systems.

## 1. Imports and load
Run from the project root in Jupyter, or upload the module and CSV to Colab.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'loan_trend_analysis.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from loan_trend_analysis import make_sample_data, clean_data, summarize

sample_path = PROJECT_ROOT / 'data' / 'loan_transactions_sample.csv'
raw = pd.read_csv(sample_path) if sample_path.exists() else make_sample_data()
print(f'Raw rows: {len(raw):,}')
raw.head()

## 2. Cleaning and data quality
Parse dates and amounts, normalize labels, remove duplicate IDs, and exclude incomplete or invalid rows.

In [ ]:
loans, quality = clean_data(raw)
quality, loans.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
loans.dtypes

## 3. Exploratory data analysis
Inspect transaction mix, successful disbursement amount, and time coverage.

In [ ]:
loans[['transaction_type','status','channel','loan_type']].describe(include='all')

In [ ]:
tables = summarize(loans)
monthly = tables['monthly']
plt.figure(figsize=(13, 4))
sns.lineplot(data=monthly, x='month', y='disbursed_amount')
plt.title('Monthly successful loan disbursement amount')
plt.xlabel('Month'); plt.ylabel('Amount (INR)'); plt.xticks(rotation=30); plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=tables['loan_type'], x='loan_type', y='disbursed_amount')
plt.title('Successful disbursed amount by loan type')
plt.ylabel('Amount (INR)'); plt.xticks(rotation=20); plt.tight_layout()

## 4. KPIs and insights
Transaction success rate is distinct from disbursement count. The delinquency field is treated cautiously because the dataset is transaction-grain.

In [ ]:
is_disb = loans['is_disbursement']
kpis = {
    'clean_transactions': len(loans),
    'successful_disbursements': int(is_disb.sum()),
    'disbursed_amount_inr': float(loans.loc[is_disb, 'amount'].sum()),
    'average_disbursement_inr': float(loans.loc[is_disb, 'amount'].mean()),
    'transaction_success_rate_pct': float(loans['is_success'].mean() * 100),
}
pd.Series(kpis)

In [ ]:
tables['loan_type']

## 5. Interpretation and limitations
A credit transaction is used as the disbursement proxy because the suggested schema has no separate booking event. Validate reversals, booking semantics, and delinquency definitions before using real data. Sample values are synthetic and are not evidence about an actual bank or borrower population.